# Desigualdades en hospitalización compleja en la red pública de salud

> Pregunta central
¿Cuál es la relación entre los altos índices de mortalidad hospitalaria y los patrones de derivación de pacientes en la red pública de salud, y qué características clínicas o de gestión explican el comportamiento de los hospitales con mayor emisión de traslados?

> Objetivos analíticos
- Evaluar diferencias regionales en casos de alta severidad hospitalaria.
- Detectar concentración de pacientes complejos en hospitales específicos.
- Relacionar severidad/mortalidad con patrones de traslado.
- Georreferenciar resultados para análisis territorial (región, comuna y hospital).

> Flujo del notebook
1. Carga y consolidación de bases GRD.
2. Limpieza y construcción de variables (año 2024 y alta severidad).
3. Agregación por región, comuna y hospital.
4. Emparejamiento geográfico con GeoJSON y base de establecimientos.
5. Visualización interactiva y auditorías de calidad de match.

## Sección 1: Carga de librerías
La siguiente celda importa las bibliotecas base para manipulación de datos, visualización y lectura robusta de archivos. Es el punto de partida para todo el pipeline de análisis.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob as gb
import os
import csv
import rapidfuzz

c:\Users\bvial\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\bvial\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## Sección 2: Consolidación de archivos GRD
La siguiente celda define columnas de interés, detecta los archivos anuales y los une en un solo DataFrame. También aplica una lectura robusta con distintos encodings para evitar pérdida de información por codificación.

In [2]:
# Definimos las columnas que realmente nos interesan para evitar cargar datos innecesarios
columnas_necesarias = [
    'CIP_ENCRIPTADO', 'COD_HOSPITAL', 'COMUNA', 'PROVINCIA', 'TIPO_PROCEDENCIA', 'FECHA_INGRESO', 'FECHAALTA', 'TIPOALTA', 'DIAGNOSTICO1', 'PROCEDIMIENTO1', 'USOSPABELLON', 'IR_29301_PESO', 'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD', 'HOSPPROCEDENCIA'
]

# 1. Buscamos todos los archivos .csv dentro de la carpeta llamada 'csv'
ruta_archivos = 'csv\*.txt'
lista_archivos = gb.glob(ruta_archivos)

# Lista vacía donde guardaremos cada DataFrame temporalmente
dataframes = []

print(f"Se encontraron {len(lista_archivos)} archivos. Comenzando la lectura...\n")

# El orden del algoritmo es importante, algunos archivos pueden ser leidos con UTF-16, otros con UTF-8-SIG, y algunos podrían requerir Latin-1.
# Si leemos los archivos del 2022 y 2023 con latin-1 se devuelve un df lleno de NaN y columnas con nombres desconocidas
# 2024 TIENE que ser leído con latin-1 para que se reconozcan las columnas correctamente, pero el resto de los archivos pueden ser leidos con UTF-16 o UTF-8-SIG

# 2. Iteramos sobre cada archivo encontrado
for archivo in lista_archivos:
    print(f"Procesando: {archivo}...")
    
    # Usamos la configuración robusta que funcionó
    try:
        df_temp = pd.read_csv(
            archivo, 
            sep='|', 
            decimal=',',
            encoding='utf-16',          
            quoting=csv.QUOTE_NONE,
            usecols=columnas_necesarias,  
            index_col=False,            
            on_bad_lines='warn',        
            low_memory=False
        )
    except UnicodeError:
        try:
            # Plan B por si algún archivo en la carpeta tiene codificación distinta
            df_temp = pd.read_csv(
                archivo, 
                sep='|', 
                decimal=',',
                encoding='utf-8-sig',       
                quoting=csv.QUOTE_NONE,
                usecols=columnas_necesarias, 
                index_col=False,
                on_bad_lines='warn',
                low_memory=False
            )
        except UnicodeDecodeError:
            #Plan C por si ninguna codificación funciona, intentamos con 'latin-1' 
            df_temp = pd.read_csv(
                archivo, 
                sep='|', 
                decimal=',',
                encoding='latin-1',       
                quoting=csv.QUOTE_NONE,
                usecols=columnas_necesarias, 
                index_col=False,
                on_bad_lines='warn',
                low_memory=False
            )
    
    # Agregamos el DataFrame a nuestra lista
    dataframes.append(df_temp)

# 3. Concatenamos todo de una sola vez
# ignore_index=True es clave: resetea el conteo de filas (0, 1, 2...) para que no se repitan
df_consolidado = pd.concat(dataframes, ignore_index=True)

print("\n¡Proceso terminado!")
print(f"El DataFrame final consolidado tiene {df_consolidado.shape[0]} filas y {df_consolidado.shape[1]} columnas.")

Se encontraron 6 archivos. Comenzando la lectura...

Procesando: csv\GRD_PUBLICO_2019.txt...
Procesando: csv\GRD_PUBLICO_2020.txt...
Procesando: csv\GRD_PUBLICO_2021.txt...
Procesando: csv\GRD_PUBLICO_2022.txt...
Procesando: csv\GRD_PUBLICO_2023.txt...
Procesando: csv\GRD_PUBLICO_2024.txt...

¡Proceso terminado!
El DataFrame final consolidado tiene 5808536 filas y 15 columnas.


### Ajustado de tipo de datos

Se dejó comentada la transformación de los datos porque más adelante se hace una comparación con una base de datos externa del gobierno para poder graficar exactamente las ubicaciones de los hospitales. Si se ejecuta esta transformación los gráficos posteriores no se visualizan.

def ajustar(df):
    # Convertir las columnas de fecha a formato datetime
    df['FECHA_INGRESO'] = pd.to_datetime(df['FECHA_INGRESO'], errors='coerce')
    df['FECHAALTA'] = pd.to_datetime(df['FECHAALTA'], errors='coerce')
    
    # Calcular la duración de la hospitalización en días
    df['DURACION_HOSPITALIZACION'] = (df['FECHAALTA'] - df['FECHA_INGRESO']).dt.days
    
    # Reemplazar valores nulos en 'DURACION_HOSPITALIZACION' con la mediana
    mediana_duracion = df['DURACION_HOSPITALIZACION'].median()
    #df['DURACION_HOSPITALIZACION'].fillna(mediana_duracion, inplace=True)
    
    # Convertir 'IR_29301_PESO', 'IR_29301_SEVERIDAD' y 'IR_29301_MORTALIDAD' a numérico, reemplazando errores con NaN
    df['CIP_ENCRIPTADO'] = pd.to_numeric(df['CIP_ENCRIPTADO'], errors='coerce')
    df['IR_29301_PESO'] = pd.to_numeric(df['IR_29301_PESO'], errors='coerce')
    df['IR_29301_SEVERIDAD'] = pd.to_numeric(df['IR_29301_SEVERIDAD'], errors='coerce')
    df['IR_29301_MORTALIDAD'] = pd.to_numeric(df['IR_29301_MORTALIDAD'], errors='coerce')
    
    # Reemplazar valores nulos en estas columnas con la mediana
    #df['IR_29301_PESO'].fillna(df['IR_29301_PESO'].median(), inplace=True)
    #df['IR_29301_SEVERIDAD'].fillna(df['IR_29301_SEVERIDAD'].median(), inplace=True)
    #df['IR_29301_MORTALIDAD'].fillna(df['IR_29301_MORTALIDAD'].median(), inplace=True)
    
    return df

# Aplicamos la función de ajuste al DataFrame consolidado
df_consolidado = ajustar(df_consolidado)

In [3]:
for i in df_consolidado.columns:
    print(f"Columna: {i} - Tipo de dato: {df_consolidado[i].dtype}")

Columna: COD_HOSPITAL - Tipo de dato: int64
Columna: CIP_ENCRIPTADO - Tipo de dato: object
Columna: PROVINCIA - Tipo de dato: str
Columna: COMUNA - Tipo de dato: str
Columna: TIPO_PROCEDENCIA - Tipo de dato: str
Columna: FECHA_INGRESO - Tipo de dato: str
Columna: FECHAALTA - Tipo de dato: str
Columna: TIPOALTA - Tipo de dato: str
Columna: DIAGNOSTICO1 - Tipo de dato: str
Columna: PROCEDIMIENTO1 - Tipo de dato: str
Columna: USOSPABELLON - Tipo de dato: object
Columna: IR_29301_PESO - Tipo de dato: object
Columna: IR_29301_SEVERIDAD - Tipo de dato: object
Columna: IR_29301_MORTALIDAD - Tipo de dato: object
Columna: HOSPPROCEDENCIA - Tipo de dato: str


## Sección 3: Revisión inicial del consolidado
La siguiente celda realiza una inspección rápida del resultado (primeras filas) para validar que el consolidado cargó correctamente antes de avanzar al preprocesamiento.

In [4]:
display(df_consolidado.head())

,COD_HOSPITAL,CIP_ENCRIPTADO,PROVINCIA,COMUNA,TIPO_PROCEDENCIA,FECHA_INGRESO,FECHAALTA,TIPOALTA,DIAGNOSTICO1,PROCEDIMIENTO1,USOSPABELLON,IR_29301_PESO,IR_29301_SEVERIDAD,IR_29301_MORTALIDAD,HOSPPROCEDENCIA
0,118100,1314867,CONCEPCION,SAN PEDRO DE LA PAZ,"CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO A...",2019-02-28,2019-03-05,DOMICILIO,O82.0,74.1,1,"0,5744",2,1,NaN
1,118100,1354418,CONCEPCION,SAN PEDRO DE LA PAZ,SERVICIO EMERGENCIA (DOMICILIO),2019-02-28,2019-03-04,DOMICILIO,O26.9,88.78,NaN,"0,2951",1,1,NaN
2,118100,239861,CONCEPCION,CONCEPCIÓN,SERVICIO EMERGENCIA (DOMICILIO),2019-02-28,2019-03-06,DOMICILIO,K92.0,87.03,NaN,"0,6736",2,2,NaN
3,118100,330326,CONCEPCION,CONCEPCIÓN,SERVICIO EMERGENCIA (DOMICILIO),2019-02-28,2019-03-02,DOMICILIO,A08.3,90.92,NaN,"0,5475",2,2,NaN
4,118100,1369293,CONCEPCION,FLORIDA,SERVICIO EMERGENCIA (DOMICILIO),2019-02-28,2019-03-06,DOMICILIO,O60.0,75.34,NaN,"0,3107",1,1,NaN


## Sección 4: Análisis Exploratorio de Datos (EDA)
A continuación, realizamos un análisis exploratorio que permite entender la estructura, distribuciones y patrones clave en los datos consolidados. Este EDA facilita la identificación de valores anómalos, relaciones entre variables y la validez de los indicadores de severidad y mortalidad antes de proceder al análisis geográfico.

In [5]:
# 1. Estadísticas descriptivas básicas
print("=" * 80)
print("ESTADÍSTICAS DESCRIPTIVAS - INDICADORES DE SEVERIDAD Y MORTALIDAD")
print("=" * 80)

print("\nSeveridad (IR_29301_SEVERIDAD):")
print(df_consolidado['IR_29301_SEVERIDAD'].describe())

print("\nMortalidad (IR_29301_MORTALIDAD):")
print(df_consolidado['IR_29301_MORTALIDAD'].describe())

print("\nPeso (IR_29301_PESO):")
print(df_consolidado['IR_29301_PESO'].describe())

ESTADÍSTICAS DESCRIPTIVAS - INDICADORES DE SEVERIDAD Y MORTALIDAD

Severidad (IR_29301_SEVERIDAD):
count     5808536
unique          9
top             1
freq      1707963
Name: IR_29301_SEVERIDAD, dtype: object

Mortalidad (IR_29301_MORTALIDAD):
count     5808536
unique          9
top             1
freq      2155428
Name: IR_29301_MORTALIDAD, dtype: object

Peso (IR_29301_PESO):
count     5808536
unique       1998
top        0,4384
freq       183744
Name: IR_29301_PESO, dtype: object


In [6]:
# 2. Análisis de completitud de datos (missing values)
print("\n" + "=" * 80)
print("ANÁLISIS DE COMPLETITUD DE DATOS")
print("=" * 80)

completitud = pd.DataFrame({
    'Columna': df_consolidado.columns,
    'No Nulos': df_consolidado.count(),
    'Nulos': df_consolidado.isnull().sum(),
    '% Completitud': (df_consolidado.count() / len(df_consolidado) * 100).round(2)
})

print(completitud.to_string(index=False))


ANÁLISIS DE COMPLETITUD DE DATOS
            Columna  No Nulos   Nulos  % Completitud
       COD_HOSPITAL   5808536       0         100.00
     CIP_ENCRIPTADO   5806492    2044          99.96
          PROVINCIA   5808536       0         100.00
             COMUNA   5808536       0         100.00
   TIPO_PROCEDENCIA   5808536       0         100.00
      FECHA_INGRESO   5808536       0         100.00
          FECHAALTA   5808518      18         100.00
           TIPOALTA   5808536       0         100.00
       DIAGNOSTICO1   5808456      80         100.00
     PROCEDIMIENTO1   5790199   18337          99.68
       USOSPABELLON   2961309 2847227          50.98
      IR_29301_PESO   5808536       0         100.00
 IR_29301_SEVERIDAD   5808536       0         100.00
IR_29301_MORTALIDAD   5808536       0         100.00
    HOSPPROCEDENCIA    703854 5104682          12.12


In [7]:
# 3. Distribución por tipo de alta
print("\n" + "=" * 80)
print("ANÁLISIS DE TIPOS DE ALTA")
print("=" * 80)

print("\nTipos de alta (TIPOALTA):")
print(df_consolidado['TIPOALTA'].value_counts(dropna=False))
print(f"\nTotal de registros: {len(df_consolidado)}")
print(f"Tipos de alta únicos: {df_consolidado['TIPOALTA'].nunique()}")


ANÁLISIS DE TIPOS DE ALTA

Tipos de alta (TIPOALTA):
TIPOALTA
DOMICILIO                                        5205550
FALLECIDO                                         170692
HOSPITALIZACIÓN DOMICILIARIA                      102168
DERIVACIÓN OTRO HOSPITAL DEL SERVICIO              98362
ALTA VOLUNTARIA                                    58281
HOSPITALIZACI�N DOMICILIARIA                       36732
DERIVACIÓN OTRO HOSPITAL DE LA RED NACIONAL        36491
DERIVACI�N OTRO HOSPITAL DEL SERVICIO              24923
FUGA DEL PACIENTE                                  19160
DERIVACIÓN A OTROS CENTROS (CÁRCEL, HOGAR DE       18296
DERIVACIÓN INST. PRIVADA (COMPRA DE SERVICIOS      15788
DERIVACI�N OTRO HOSPITAL DE LA RED NACIONAL         7938
DERIVACIÓN INST. PRIVADA (VOLUNTARIO)               6317
DERIVACI�N A OTROS CENTROS (C�RCEL, HOGAR DE        3745
DERIVACI�N INST. PRIVADA (COMPRA DE SERVICIOS       2865
DERIVACI�N INST. PRIVADA (VOLUNTARIO)               1126
NO IDENTIFICADA          

In [8]:
# 4. Correlación entre indicadores de severidad
print("\n" + "=" * 80)
print("MATRIZ DE CORRELACIÓN - INDICADORES CLAVE")
print("=" * 80)

df_numeric = df_consolidado[['IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD', 'IR_29301_PESO']].copy()
df_numeric['IR_29301_SEVERIDAD'] = pd.to_numeric(df_numeric['IR_29301_SEVERIDAD'], errors='coerce')
df_numeric['IR_29301_MORTALIDAD'] = pd.to_numeric(df_numeric['IR_29301_MORTALIDAD'], errors='coerce')
df_numeric['IR_29301_PESO'] = (
    df_numeric['IR_29301_PESO']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .apply(lambda x: pd.to_numeric(x, errors='coerce') if x else None)
)

print("\nCorrelaciones:")
print(df_numeric.corr())


MATRIZ DE CORRELACIÓN - INDICADORES CLAVE

Correlaciones:
                     IR_29301_SEVERIDAD  IR_29301_MORTALIDAD  IR_29301_PESO
IR_29301_SEVERIDAD             1.000000             0.862053       0.436801
IR_29301_MORTALIDAD            0.862053             1.000000       0.420835
IR_29301_PESO                  0.436801             0.420835       1.000000


In [9]:
df_consolidado.head()


,COD_HOSPITAL,CIP_ENCRIPTADO,PROVINCIA,COMUNA,TIPO_PROCEDENCIA,FECHA_INGRESO,FECHAALTA,TIPOALTA,DIAGNOSTICO1,PROCEDIMIENTO1,USOSPABELLON,IR_29301_PESO,IR_29301_SEVERIDAD,IR_29301_MORTALIDAD,HOSPPROCEDENCIA
0,118100,1314867,CONCEPCION,SAN PEDRO DE LA PAZ,"CENTRO ESPECIALIDADES (CDT, CRS, CONSULTORIO A...",2019-02-28,2019-03-05,DOMICILIO,O82.0,74.1,1,"0,5744",2,1,NaN
1,118100,1354418,CONCEPCION,SAN PEDRO DE LA PAZ,SERVICIO EMERGENCIA (DOMICILIO),2019-02-28,2019-03-04,DOMICILIO,O26.9,88.78,NaN,"0,2951",1,1,NaN
2,118100,239861,CONCEPCION,CONCEPCIÓN,SERVICIO EMERGENCIA (DOMICILIO),2019-02-28,2019-03-06,DOMICILIO,K92.0,87.03,NaN,"0,6736",2,2,NaN
3,118100,330326,CONCEPCION,CONCEPCIÓN,SERVICIO EMERGENCIA (DOMICILIO),2019-02-28,2019-03-02,DOMICILIO,A08.3,90.92,NaN,"0,5475",2,2,NaN
4,118100,1369293,CONCEPCION,FLORIDA,SERVICIO EMERGENCIA (DOMICILIO),2019-02-28,2019-03-06,DOMICILIO,O60.0,75.34,NaN,"0,3107",1,1,NaN


In [10]:
df_consolidado.info()

<class 'pandas.DataFrame'>
RangeIndex: 5808536 entries, 0 to 5808535
Data columns (total 15 columns):
 #   Column               Dtype 
---  ------               ----- 
 0   COD_HOSPITAL         int64 
 1   CIP_ENCRIPTADO       object
 2   PROVINCIA            str   
 3   COMUNA               str   
 4   TIPO_PROCEDENCIA     str   
 5   FECHA_INGRESO        str   
 6   FECHAALTA            str   
 7   TIPOALTA             str   
 8   DIAGNOSTICO1         str   
 9   PROCEDIMIENTO1       str   
 10  USOSPABELLON         object
 11  IR_29301_PESO        object
 12  IR_29301_SEVERIDAD   object
 13  IR_29301_MORTALIDAD  object
 14  HOSPPROCEDENCIA      str   
dtypes: int64(1), object(5), str(9)
memory usage: 664.7+ MB


## Sección 5: Filtrado de año y limpieza de variables
La siguiente celda filtra el año 2024, convierte fechas y columnas numéricas clave, y elimina duplicados exactos. Este paso deja la base lista para indicadores de severidad comparables.

In [11]:
# 1. Nos quedaremos solo con 2024 para trabajar el mapa de calor
df_2024 = df_consolidado[
    df_consolidado['FECHA_INGRESO'].astype(str).str.startswith('2024-')
].copy()

print("Filas 2024 antes de limpiar:", len(df_2024))

# 2. La fecha esta como object, asi que se transformara
df_2024['FECHA_INGRESO'] = pd.to_datetime(
    df_2024['FECHA_INGRESO'],
    format='%Y-%m-%d',
    errors='coerce'
)

df_2024['FECHAALTA'] = pd.to_datetime(
    df_2024['FECHAALTA'],
    format='%Y-%m-%d',
    errors='coerce'
)

# 3. Convertir variables numéricas para comparar porcentajes
df_2024['IR_29301_SEVERIDAD'] = pd.to_numeric(df_2024['IR_29301_SEVERIDAD'], errors='coerce')
df_2024['IR_29301_MORTALIDAD'] = pd.to_numeric(df_2024['IR_29301_MORTALIDAD'], errors='coerce')

df_2024['IR_29301_PESO'] = (
    df_2024['IR_29301_PESO']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df_2024['IR_29301_PESO'] = pd.to_numeric(df_2024['IR_29301_PESO'], errors='coerce')

# 4. Revisar duplicados exactos
print("Duplicados exactos antes:", df_2024.duplicated().sum())

# 5. Eliminar duplicados exactos, dejando una fila
df_2024 = df_2024.drop_duplicates()

print("Filas 2024 después de limpiar:", len(df_2024))

Filas 2024 antes de limpiar: 1071512
Duplicados exactos antes: 1016
Filas 2024 después de limpiar: 1070496


## Sección 6: Construcción y validación de indicador de alta severidad
Las siguientes celdas crean la variable booleana de alta severidad y muestran distribuciones para validar que el umbral aplicado sea coherente con los datos.

In [12]:
df_2024['ALTA_SEVERIDAD'] = df_2024['IR_29301_SEVERIDAD'] >= 3

In [13]:
print(df_2024['IR_29301_SEVERIDAD'].value_counts(dropna=False).sort_index())


print(df_2024['ALTA_SEVERIDAD'].value_counts(dropna=False))

IR_29301_SEVERIDAD
0.0    211033
1.0    384084
2.0    264835
3.0    210530
NaN        14
Name: count, dtype: int64
ALTA_SEVERIDAD
False    859966
True     210530
Name: count, dtype: int64


In [14]:
print("Provincias únicas:", df_2024['PROVINCIA'].nunique())
print("Comunas únicas:", df_2024['COMUNA'].nunique())

print(df_2024['PROVINCIA'].value_counts().head(20))

Provincias únicas: 57
Comunas únicas: 344
PROVINCIA
SANTIAGO       244584
CONCEPCION      76685
CAUTIN          58340
VALPARAISO      43642
CORDILLERA      40945
ELQUI           37334
BIO-BIO         30697
TALCA           30539
LLANQUIHUE      28481
CACHAPOAL       26622
MAIPO           24407
LINARES         22976
CURICO          22081
DIGUILL�N       21755
MALLECO         21084
ANTOFAGASTA     20466
ARICA           20138
OSORNO          20042
VALDIVIA        19692
IQUIQUE         19553
Name: count, dtype: int64


## Sección 7: Estandarización territorial y resumen por región
La siguiente celda normaliza provincias, asigna región y código regional, y genera un resumen con porcentaje de alta severidad por región. Es la base del análisis geográfico nacional.

In [15]:
df_2024['PROVINCIA'] = (
    df_2024['PROVINCIA']
    .astype(str)
    .str.upper()
    .str.strip()
)

mapa_provincia_region = {
    'ARICA': 'ARICA Y PARINACOTA',
    'PARINACOTA': 'ARICA Y PARINACOTA',
    'IQUIQUE': 'TARAPACA',
    'TAMARUGAL': 'TARAPACA',
    'ANTOFAGASTA': 'ANTOFAGASTA',
    'EL LOA': 'ANTOFAGASTA',
    'TOCOPILLA': 'ANTOFAGASTA',
    'CHAÑARAL': 'ATACAMA',
    'COPIAPO': 'ATACAMA',
    'HUASCO': 'ATACAMA',
    'ELQUI': 'COQUIMBO',
    'LIMARI': 'COQUIMBO',
    'CHOAPA': 'COQUIMBO',
    'VALPARAISO': 'VALPARAISO',
    'SAN ANTONIO': 'VALPARAISO',
    'QUILLOTA': 'VALPARAISO',
    'PETORCA': 'VALPARAISO',
    'LOS ANDES': 'VALPARAISO',
    'SAN FELIPE': 'VALPARAISO',
    'ISLA DE PASCUA': 'VALPARAISO',
    'SANTIAGO': 'METROPOLITANA',
    'CORDILLERA': 'METROPOLITANA',
    'MAIPO': 'METROPOLITANA',
    'MELIPILLA': 'METROPOLITANA',
    'TALAGANTE': 'METROPOLITANA',
    'CACHAPOAL': 'O’HIGGINS',
    'COLCHAGUA': 'O’HIGGINS',
    'CARDENAL CARO': 'O’HIGGINS',
    'CURICO': 'MAULE',
    'TALCA': 'MAULE',
    'LINARES': 'MAULE',
    'CAUQUENES': 'MAULE',
    'DIGUILLIN': 'ÑUBLE',
    'PUNILLA': 'ÑUBLE',
    'ITATA': 'ÑUBLE',
    'CONCEPCION': 'BIOBIO',
    'BIO-BIO': 'BIOBIO',
    'ARAUCO': 'BIOBIO',
    'MALLECO': 'LA ARAUCANIA',
    'CAUTIN': 'LA ARAUCANIA',
    'VALDIVIA': 'LOS RIOS',
    'RANCO': 'LOS RIOS',
    'OSORNO': 'LOS LAGOS',
    'LLANQUIHUE': 'LOS LAGOS',
    'CHILOE': 'LOS LAGOS',
    'PALENA': 'LOS LAGOS',
    'AYSEN': 'AYSEN',
    'COYHAIQUE': 'AYSEN',
    'CAPITAN PRAT': 'AYSEN',
    'GENERAL CARRERA': 'AYSEN',
    'MAGALLANES': 'MAGALLANES',
    'ULTIMA ESPERANZA': 'MAGALLANES',
    'TIERRA DEL FUEGO': 'MAGALLANES',
    'ANTARTICA': 'MAGALLANES'
}

df_2024['REGION'] = df_2024['PROVINCIA'].map(mapa_provincia_region)

df_2024['PROVINCIA'] = df_2024['PROVINCIA'].replace({
    'DIGUILL�N': 'DIGUILLIN',
    'DIGUILLÍN': 'DIGUILLIN'
})

mapa_region_codigo = {
    'ARICA Y PARINACOTA': 15,
    'TARAPACA': 1,
    'ANTOFAGASTA': 2,
    'ATACAMA': 3,
    'COQUIMBO': 4,
    'VALPARAISO': 5,
    'METROPOLITANA': 13,
    'O’HIGGINS': 6,
    "O'HIGGINS": 6,
    'MAULE': 7,
    'ÑUBLE': 16,
    'BIOBIO': 8,
    'LA ARAUCANIA': 9,
    'LOS RIOS': 14,
    'LOS LAGOS': 10,
    'AYSEN': 11,
    'MAGALLANES': 12
}

df_2024['codregion'] = df_2024['REGION'].map(mapa_region_codigo)

resumen_region = df_2024.groupby('REGION').agg(
    total=('CIP_ENCRIPTADO', 'count'),
    alta=('ALTA_SEVERIDAD', 'sum')
).reset_index()

resumen_region['porcentaje'] = resumen_region['alta'] / resumen_region['total'] * 100
resumen_region['codregion'] = resumen_region['REGION'].map(mapa_region_codigo)

print(df_2024[['PROVINCIA', 'REGION', 'codregion']].head())
print(resumen_region.sort_values('porcentaje', ascending=False))

           PROVINCIA         REGION  codregion
4722723  MARGA MARGA            NaN        NaN
4722724   VALPARAISO     VALPARAISO        5.0
4722725        ELQUI       COQUIMBO        4.0
4722726     SANTIAGO  METROPOLITANA       13.0
4722727   CONCEPCION         BIOBIO        8.0
                REGION   total   alta  porcentaje  codregion
11       METROPOLITANA  339425  77321   22.779996         13
14          VALPARAISO   94595  20567   21.742164          5
12           O’HIGGINS   44692   9125   20.417524          6
4               BIOBIO  117156  23450   20.016047          8
9           MAGALLANES   13427   2668   19.870410         12
7            LOS LAGOS   58867  11294   19.185622         10
15               ÑUBLE   13714   2532   18.462885         16
10               MAULE   81911  14409   17.591044          7
8             LOS RIOS   23547   4013   17.042511         14
6         LA ARAUCANIA   79424  13142   16.546636          9
0          ANTOFAGASTA   33581   5435   16.1847

## Sección 8: Dependencias geoespaciales
La siguiente celda instala librerías para visualización geográfica e interactividad. Debe ejecutarse solo si el entorno aún no tiene estos paquetes.

In [16]:
!pip install plotly geopandas

## Sección 9: Carga de GeoJSON y visualización regional inicial
Las siguientes celdas cargan la geometría de regiones de Chile, validan el resumen regional y construyen una visualización exploratoria para comparar porcentaje de alta severidad entre regiones.

In [17]:
import json
import urllib.request

url = "https://raw.githubusercontent.com/caracena/chile-geojson/master/regiones.json"

with urllib.request.urlopen(url) as response:
    geojson_chile = json.load(response)

In [18]:
mapa_region_codigo = {
    'ARICA Y PARINACOTA': 15,
    'TARAPACA': 1,
    'ANTOFAGASTA': 2,
    'ATACAMA': 3,
    'COQUIMBO': 4,
    'VALPARAISO': 5,
    'METROPOLITANA': 13,
    'O’HIGGINS': 6,
    "O'HIGGINS": 6,
    'MAULE': 7,
    'ÑUBLE': 16,
    'BIOBIO': 8,
    'LA ARAUCANIA': 9,
    'LOS RIOS': 14,
    'LOS LAGOS': 10,
    'AYSEN': 11,
    'MAGALLANES': 12
}

resumen_region['codregion'] = resumen_region['REGION'].map(mapa_region_codigo)

In [19]:
resumen_region.sort_values('porcentaje', ascending=False)

,REGION,total,alta,porcentaje,codregion
11,METROPOLITANA,339425,77321,22.779996,13
14,VALPARAISO,94595,20567,21.742164,5
12,O’HIGGINS,44692,9125,20.417524,6
4,BIOBIO,117156,23450,20.016047,8
9,MAGALLANES,13427,2668,19.870410,12
7,LOS LAGOS,58867,11294,19.185622,10
15,ÑUBLE,13714,2532,18.462885,16
10,MAULE,81911,14409,17.591044,7
8,LOS RIOS,23547,4013,17.042511,14
6,LA ARAUCANIA,79424,13142,16.546636,9


In [20]:
import plotly.express as px

px.bar(
    resumen_region.sort_values('porcentaje', ascending=False),
    x='REGION',
    y='porcentaje',
    title='Ranking de complejidad por región'
)

## Sección 10: Mapa regional interactivo con detalle hospitalario
La siguiente celda integra el mapa regional con una tabla dinámica de hospitales por región, permitiendo inspeccionar los establecimientos con mayor volumen y severidad tras seleccionar una región.

In [21]:

import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go

# 1. Cargar tabla maestra hospitales

df_maestra = pd.read_excel("csv\Tablas maestras bases GRD  (1).xlsx")

df_maestra = df_maestra.rename(columns={
    'HOSPITALES': 'COD_HOSPITAL',
    'Unnamed: 1': 'NOMBRE_HOSPITAL'
})

# Asegurar tipo numérico
df_maestra['COD_HOSPITAL'] = pd.to_numeric(df_maestra['COD_HOSPITAL'], errors='coerce')

# 2. Resumen por hospital

resumen_hospital = (
    df_2024.groupby(['REGION', 'COD_HOSPITAL'])
    .agg(
        total=('CIP_ENCRIPTADO', 'count'),
        alta=('ALTA_SEVERIDAD', 'sum')
    )
    .reset_index()
)

resumen_hospital['porcentaje'] = resumen_hospital['alta'] / resumen_hospital['total'] * 100

# Asegurar tipo para merge
resumen_hospital['COD_HOSPITAL'] = pd.to_numeric(resumen_hospital['COD_HOSPITAL'], errors='coerce')

# 3. Merge para traer nombres

resumen_hospital = resumen_hospital.merge(
    df_maestra[['COD_HOSPITAL', 'NOMBRE_HOSPITAL']],
    on='COD_HOSPITAL',
    how='left'
)


# 4. MAPA


fig = px.choropleth(
    resumen_region,
    geojson=geojson_chile,
    locations='codregion',
    featureidkey='properties.codregion',
    color='porcentaje',
    color_continuous_scale='Reds',
    hover_name='REGION',
    custom_data=['REGION', 'total', 'alta'],
    title='Porcentaje de pacientes con alta severidad por región, Chile 2024'
)

fig.update_geos(
    fitbounds="locations",
    visible=False
)

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.update_traces(
    hovertemplate=
    "<b>%{customdata[0]}</b><br>" +
    "Total pacientes: %{customdata[1]}<br>" +
    "Alta severidad: %{customdata[2]}<br>" +
    "Porcentaje: %{z:.2f}%<extra></extra>"
)

figw = go.FigureWidget(fig)


#Esta linea de codigo sirve para filtrar los hospitales con menos casos
#resumen_hospital = resumen_hospital[resumen_hospital['total'] >= 30]

# 5. SALIDA INTERACTIVA

salida = widgets.Output()

def mostrar_hospitales(region_seleccionada):
    with salida:
        clear_output()
        print(f"Región seleccionada: {region_seleccionada}")
        
        tabla = (
            resumen_hospital[resumen_hospital['REGION'] == region_seleccionada]
            .sort_values(['porcentaje', 'alta', 'total'], ascending=[False, False, False])
            [['NOMBRE_HOSPITAL', 'total', 'alta', 'porcentaje']]  # 👈 NOMBRE AQUÍ
            .copy()
        )
        
        tabla['porcentaje'] = tabla['porcentaje'].round(2)
        
        display(tabla.head(20))

# 6. CLICK EN MAPA

def click_region(trace, points, selector):
    if points.point_inds:
        idx = points.point_inds[0]
        region_seleccionada = resumen_region.iloc[idx]['REGION']
        mostrar_hospitales(region_seleccionada)

figw.data[0].on_click(click_region)

# 7. MOSTRAR

display(figw, salida)

FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'customdata': array([['ANTOFAGASTA', 33581, 5435],
                                   ['ARICA Y PARINACOTA', 20144, 2580],
                                   ['ATACAMA', 20495, 2975],
                                   ['AYSEN', 906, 106],
                                   ['BIOBIO', 117156, 23450],
                                   ['COQUIMBO', 51412, 7966],
                                   ['LA ARAUCANIA', 79424, 13142],
                                   ['LOS LAGOS', 58867, 11294],
                                   ['LOS RIOS', 23547, 4013],
                                   ['MAGALLANES', 13427, 2668],
                                   ['MAULE', 81911, 14409],
                                   ['METROPOLITANA', 339425, 77321],
                                   ['O’HIGGINS', 44692, 9125],
                                   ['TARAPACA', 20838, 3158],
                                   ['VALPARAISO', 94595,

Output()

## Sección 11: Preparación de análisis comunal
A partir del GeoJSON de comunas, se construirá la capa territorial de mayor detalle para analizar heterogeneidad intrarregional y, posteriormente, superponer hospitales.

In [22]:
import json
import urllib.request
import plotly.express as px

# 1. Cargar geojson de comunas por región
geojson_regiones = {}

base_url = "https://raw.githubusercontent.com/caracena/chile-geojson/master/"

for i in range(1, 17):
    url = f"{base_url}{i}.geojson"
    try:
        with urllib.request.urlopen(url) as response:
            geojson_regiones[i] = json.load(response)
        print(f"✔ Región {i} cargada")
    except Exception as e:
        print(f"❌ Error en región {i}: {e}")

# 2. Resumen por comuna
resumen_comuna = (
    df_2024.groupby(['REGION', 'codregion', 'COMUNA'])
    .agg(
        total=('CIP_ENCRIPTADO', 'count'),
        alta=('ALTA_SEVERIDAD', 'sum')
    )
    .reset_index()
)

resumen_comuna['porcentaje'] = resumen_comuna['alta'] / resumen_comuna['total'] * 100

resumen_comuna['COMUNA'] = resumen_comuna['COMUNA'].astype(str).str.strip()

print(resumen_comuna.columns.tolist())
display(resumen_comuna.head())

resumen_comuna['porcentaje'] = resumen_comuna['alta'] / resumen_comuna['total'] * 100

# 3. Limpiar nombres de comuna
resumen_comuna['COMUNA'] = (
    resumen_comuna['COMUNA']
    .astype(str)    
    .str.strip()
)



# 4. Función para mostrar mapa comunal
def mostrar_mapa_comunas(region_seleccionada, codregion):
    geojson_region = geojson_regiones.get(codregion)
    
    if geojson_region is None:
        print(f"No se encontró GeoJSON para la región {codregion}")
        return
    
    tabla_comunas = resumen_comuna[resumen_comuna['REGION'] == region_seleccionada].copy()
    
    # Ajuste simple para empatar con nombres del geojson
    tabla_comunas['COMUNA_MAPA'] = tabla_comunas['COMUNA'].str.title().str.strip()

    fig_comunas = px.choropleth(
        tabla_comunas,
        geojson=geojson_region,
        locations='COMUNA_GEOJSON',
        featureidkey='properties.Comuna',
        color='porcentaje',
        color_continuous_scale='OrRd',
        range_color=(0, 50),
        hover_name='COMUNA',
        custom_data=['total', 'alta'],
        title=f'Porcentaje de pacientes con alta severidad por comuna, {region_seleccionada} - 2024'
    )

    fig_comunas.update_geos(
        fitbounds="locations",
        visible=False
    )

    fig_comunas.update_layout(
    margin={"r": 20, "t": 60, "l": 20, "b": 20},
    paper_bgcolor="#F4F8FF",
    plot_bgcolor="#F4F8FF",
    title={
        "text": f"Porcentaje de pacientes con alta severidad por comuna, {region_seleccionada} - 2024",
        "x": 0.5,
        "xanchor": "center"
    }
)

    fig_comunas.update_traces(
        hovertemplate=
        "<b>%{hovertext}</b><br>" +
        "Total pacientes: %{customdata[0]}<br>" +
        "Alta severidad: %{customdata[1]}<br>" +
        "Porcentaje: %{z:.2f}%<extra></extra>"
    )

    fig_comunas.show()

✔ Región 1 cargada
✔ Región 2 cargada
✔ Región 3 cargada
✔ Región 4 cargada
✔ Región 5 cargada
✔ Región 6 cargada
✔ Región 7 cargada
✔ Región 8 cargada
✔ Región 9 cargada
✔ Región 10 cargada
✔ Región 11 cargada
✔ Región 12 cargada
✔ Región 13 cargada
✔ Región 14 cargada
✔ Región 15 cargada
✔ Región 16 cargada
['REGION', 'codregion', 'COMUNA', 'total', 'alta', 'porcentaje']


,REGION,codregion,COMUNA,total,alta,porcentaje
0,ANTOFAGASTA,2.0,ALTO HOSPICIO,1,0,0.000000
1,ANTOFAGASTA,2.0,ANTOFAGASTA,19195,2830,14.743423
2,ANTOFAGASTA,2.0,CALAMA,11192,2082,18.602573
3,ANTOFAGASTA,2.0,CAMI�A,1,1,100.000000
4,ANTOFAGASTA,2.0,CA�ETE,1,0,0.000000


## Sección 12: Emparejamiento de nombres de comunas (exacto y fuzzy)
La siguiente celda normaliza nombres y resuelve diferencias entre fuentes (base analítica vs GeoJSON) mediante reglas manuales y fuzzy matching, dejando trazabilidad del tipo y score de match.

In [23]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import process, fuzz

# 1. Función para normalizar texto
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).upper().strip()
    texto = ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )
    texto = re.sub(r'[^A-Z0-9 ]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


# 2. Crear tabla maestra de comunas desde geojson

filas_geojson = []

for codregion, geojson_region in geojson_regiones.items():
    for feature in geojson_region['features']:
        comuna_geo = feature['properties'].get('Comuna', None)
        filas_geojson.append({
            'codregion_geo': codregion,
            'COMUNA_GEOJSON': comuna_geo
        })

comunas_geojson_df = pd.DataFrame(filas_geojson)

# normalizar comuna geojson
comunas_geojson_df['COMUNA_NORM'] = comunas_geojson_df['COMUNA_GEOJSON'].apply(normalizar_texto)

# 3. Normalizar comunas del resumen
resumen_comuna['COMUNA_NORM'] = resumen_comuna['COMUNA'].apply(normalizar_texto)
resumen_comuna['REGION_NORM'] = resumen_comuna['REGION'].apply(normalizar_texto)

equivalencias_comunas = {
    'CAMI A': 'CAMINA',
    'LOS NGELES': 'LOS ANGELES',
    'CHA ARAL': 'CHANARAL',
    'VICU A': 'VICUNA',
    'R O IB EZ': 'RIO IBANEZ',
    'PUR N': 'PUREN',
    'TIR A': 'TIRUA',
    'AIS N': 'AISEN',
    'PE ALOL N': 'PENALOLEN',
    'PUC N': 'PUCON',
    'TOLT N': 'TOLTEN',
    'VILC N': 'VILCUN',
    'U OA': 'NUNOA',
    'M FIL': 'MAFIL',
    'PE AFLOR': 'PENAFLOR',
    'REQU NOA': 'REQUINOA',
    'JUAN FERN NDEZ': 'JUAN FERNANDEZ'
}

resumen_comuna['COMUNA_NORM'] = resumen_comuna['COMUNA_NORM'].replace(equivalencias_comunas)

# 4. Merge exacto SOLO por comuna normalizada
resumen_comuna_match = resumen_comuna.merge(
    comunas_geojson_df[['codregion_geo', 'COMUNA_GEOJSON', 'COMUNA_NORM']],
    on='COMUNA_NORM',
    how='left'
)

# marcar estado inicial
resumen_comuna_match['TIPO_MATCH'] = 'exacto'
resumen_comuna_match.loc[resumen_comuna_match['COMUNA_GEOJSON'].isna(), 'TIPO_MATCH'] = 'sin_match'
resumen_comuna_match['SCORE_MATCH'] = 100
resumen_comuna_match.loc[resumen_comuna_match['COMUNA_GEOJSON'].isna(), 'SCORE_MATCH'] = pd.NA

# 5. Fuzzy solo para comunas sin match
umbral_fuzzy = 70

candidatos_todos = comunas_geojson_df['COMUNA_NORM'].dropna().unique().tolist()

for idx, fila in resumen_comuna_match[resumen_comuna_match['COMUNA_GEOJSON'].isna()].iterrows():
    comuna_objetivo = fila['COMUNA_NORM']

    if not comuna_objetivo:
        continue

    mejor = process.extractOne(
        comuna_objetivo,
        candidatos_todos,
        scorer=fuzz.token_sort_ratio
    )

    if mejor is None:
        continue

    comuna_match, score, _ = mejor

    if score >= umbral_fuzzy:
        fila_geo = comunas_geojson_df[
            comunas_geojson_df['COMUNA_NORM'] == comuna_match
        ].iloc[0]

        resumen_comuna_match.at[idx, 'COMUNA_GEOJSON'] = fila_geo['COMUNA_GEOJSON']
        resumen_comuna_match.at[idx, 'codregion_geo'] = fila_geo['codregion_geo']
        resumen_comuna_match.at[idx, 'TIPO_MATCH'] = 'fuzzy'
        resumen_comuna_match.at[idx, 'SCORE_MATCH'] = score

# 6. Usar codregion del geojson como codregion final
resumen_comuna_match['codregion_final'] = resumen_comuna_match['codregion_geo']

print("Total filas en resumen_comuna_match:", len(resumen_comuna_match))
print("Con match:", resumen_comuna_match['COMUNA_GEOJSON'].notna().sum())
print("Sin match real:", resumen_comuna_match['COMUNA_GEOJSON'].isna().sum())

display(
    resumen_comuna_match[
        resumen_comuna_match['COMUNA_GEOJSON'].isna()
    ][['REGION', 'codregion', 'COMUNA', 'COMUNA_NORM', 'TIPO_MATCH', 'SCORE_MATCH']]
    .drop_duplicates()
    .sort_values(['REGION', 'COMUNA'])
)

Total filas en resumen_comuna_match: 597
Con match: 597
Sin match real: 0


,REGION,codregion,COMUNA,COMUNA_NORM,TIPO_MATCH,SCORE_MATCH


## Sección 13: Carga de establecimientos de salud
La siguiente celda incorpora la base oficial de establecimientos para obtener coordenadas y atributos territoriales de hospitales, necesarios para georreferenciar resultados.

In [24]:
df_establecimientos = pd.read_csv(
    "csv\establecimientos_20260317.csv",
    sep=';',
    encoding='utf-8-sig'
)

print(df_establecimientos.head())
print(df_establecimientos.columns)
print(df_establecimientos.shape)

   EstablecimientoCodigo                               EstablecimientoGlosa  \
0                 126704  Hospital Comunitario Cristina Calderón de Puer...   
1                 126204                   Hospital Naval (Puerto Williams)   
2                 126412                       Posta de Salud Rural Cameron   
3                 126414                   Posta de Salud Rural Agua Fresca   
4                 126102    Hospital Dr. Marco Antonio Chamorro ( Porvenir)   

  EstablecimientoCodigoAntiguo EstablecimientoCodigoMadreAntiguo  \
0                       26-704                               NaN   
1                       26-204                               NaN   
2                       26-412                               NaN   
3                       26-414                               NaN   
4                       26-102                               NaN   

   EstablecimientoCodigoMadreNuevo  RegionCodigo  \
0                              NaN            12   
1           

In [25]:
df_establecimientos = df_establecimientos[
    df_establecimientos['TipoAtencionEstabGlosa'] == 'Atención Cerrada-Hospitalaria'
]

## Sección 14: Resumen hospitalario consolidado
La siguiente celda agrega el desempeño por hospital (total, casos de alta severidad y porcentaje) para evitar duplicidad por región y habilitar comparación nacional entre establecimientos.

In [26]:
# =========================================================
# RESUMEN HOSPITALARIO SOLO POR HOSPITAL
# =========================================================

resumen_hospital_unico = (
    resumen_hospital.groupby(['COD_HOSPITAL', 'NOMBRE_HOSPITAL'], as_index=False)
    .agg(
        total=('total', 'sum'),
        alta=('alta', 'sum')
    )
)

resumen_hospital_unico['porcentaje'] = (
    resumen_hospital_unico['alta'] / resumen_hospital_unico['total'] * 100
)

print("Filas en resumen_hospital original:", len(resumen_hospital))
print("Hospitales únicos:", len(resumen_hospital_unico))

display(
    resumen_hospital_unico.sort_values('total', ascending=False).head(20)
)

Filas en resumen_hospital original: 849
Hospitales únicos: 71


,COD_HOSPITAL,NOMBRE_HOSPITAL,total,alta,porcentaje
36,114101,Complejo Hospitalario Dr. Sótero del Río (Sant...,47796,12067,25.246883
50,118100,Hospital Clínico Regional Dr. Guillermo Grant ...,34198,8613,25.185683
43,116105,Hospital Dr. César Garavagno Burotto (Talca),33380,5684,17.028161
56,120101,Complejo Asistencial Dr. Víctor Ríos Ruiz (Los...,30088,5461,18.150093
64,124105,Hospital de Puerto Montt,29932,6168,20.606709
57,121109,Hospital Dr. Hernán Henríquez Aravena (Temuco),29020,4967,17.115782
32,113100,"Hospital Barros Luco Trudeau (Santiago, San Mi...",26589,5102,19.188386
39,115100,Hospital Regional de Rancagua,26562,6472,24.365635
19,110100,"Hospital San Juan de Dios (Santiago, Santiago)",26200,5515,21.049618
21,110120,"Hospital Dr. Félix Bulnes Cerda (Santiago, Qui...",25387,6211,24.465278


## Sección 15: Georreferenciación de hospitales y resolución de identidad
La siguiente celda aplica una estrategia en etapas para asociar hospitales con coordenadas: match exacto por nombre completo, match por nombre simplificado, equivalencias manuales y fuzzy matching con umbral.

In [27]:
import pandas as pd
import re
import unicodedata
from rapidfuzz import process, fuzz

# =========================================================
# 1. PARTIR LIMPIO
# =========================================================

rh = resumen_hospital_unico.copy()
de = df_establecimientos.copy()

columnas_borrar = [
    'NOMBRE_FULL_NORM', 'NOMBRE_SIMPLE_NORM', 'NOMBRE_MATCH', 'SCORE_MATCH',
    'NOMBRE_GEO', 'LAT_GEO', 'LON_GEO', 'REGION_GEO', 'COMUNA_GEO',
    'NOMBRE_GEO_FUZZY', 'LAT_GEO_FUZZY', 'LON_GEO_FUZZY',
    'REGION_GEO_FUZZY', 'COMUNA_GEO_FUZZY'
]
rh = rh.drop(columns=[c for c in columnas_borrar if c in rh.columns], errors='ignore')

rh = rh.loc[:, ~rh.columns.duplicated()].copy()
de = de.loc[:, ~de.columns.duplicated()].copy()

# =========================================================
# 2. FILTRAR SOLO HOSPITALES DEL DATASET GEO
# =========================================================

de = de[de['TipoAtencionEstabGlosa'] == 'Atención Cerrada-Hospitalaria'].copy()

# =========================================================
# 3. FUNCIONES DE NORMALIZACIÓN
# =========================================================

def normalizar_texto(texto):
    if pd.isna(texto):
        return None
    texto = str(texto).upper().strip()
    texto = ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )
    texto = re.sub(r'[^A-Z0-9(), ]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def normalizar_nombre_full(texto):
    t = normalizar_texto(texto)
    if t is None:
        return None
    return t

def normalizar_nombre_simple(texto):
    if pd.isna(texto):
        return None

    original = str(texto).upper().strip()

    texto = ''.join(
        c for c in unicodedata.normalize('NFKD', original)
        if not unicodedata.combining(c)
    )

    # quitar contenido entre paréntesis SOLO para la versión simple
    texto = re.sub(r'\(.*?\)', ' ', texto)

    reemplazos = {
        'HOSPITAL ': ' ',
        'HOSP. ': ' ',
        'CLINICO ': ' ',
        'CLINICA ': ' ',
        'DOCTOR ': ' ',
        'DR. ': ' ',
        'DR ': ' ',
        'DRA. ': ' ',
        'DRA ': ' ',
        'DEL ': ' ',
        'DE LA ': ' ',
        'DE LOS ': ' ',
        'DE LAS ': ' ',
        'REGIONAL ': ' ',
        'COMUNITARIO ': ' ',
        'PROVINCIAL ': ' ',
        'BASE ': ' '
    }

    for buscar, reemplazar in reemplazos.items():
        texto = texto.replace(buscar, reemplazar)

    texto = re.sub(r'[^A-Z0-9 ]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto if texto != '' else None

# =========================================================
# 4. NORMALIZAR TABLAS
# =========================================================

rh['NOMBRE_FULL_NORM'] = rh['NOMBRE_HOSPITAL'].apply(normalizar_nombre_full)
rh['NOMBRE_SIMPLE_NORM'] = rh['NOMBRE_HOSPITAL'].apply(normalizar_nombre_simple)

de['NOMBRE_FULL_NORM'] = de['EstablecimientoGlosa'].apply(normalizar_nombre_full)
de['NOMBRE_SIMPLE_NORM'] = de['EstablecimientoGlosa'].apply(normalizar_nombre_simple)

# =========================================================
# 5. PREPARAR TABLA GEO
# =========================================================

de_geo = de[[
    'EstablecimientoGlosa',
    'NOMBRE_FULL_NORM',
    'NOMBRE_SIMPLE_NORM',
    'Latitud',
    'Longitud',
    'RegionGlosa',
    'ComunaGlosa'
]].copy()

de_geo = de_geo.rename(columns={
    'EstablecimientoGlosa': 'NOMBRE_GEO',
    'Latitud': 'LAT_GEO',
    'Longitud': 'LON_GEO',
    'RegionGlosa': 'REGION_GEO',
    'ComunaGlosa': 'COMUNA_GEO'
})

# para match exacto por nombre completo
de_geo_full = de_geo.drop_duplicates(subset=['NOMBRE_FULL_NORM']).copy()

# para match por nombre simple
de_geo_simple = de_geo.drop_duplicates(subset=['NOMBRE_SIMPLE_NORM']).copy()

# =========================================================
# 6. MATCH EXACTO POR NOMBRE COMPLETO
# =========================================================

rh = rh.merge(
    de_geo_full[['NOMBRE_FULL_NORM', 'NOMBRE_GEO', 'LAT_GEO', 'LON_GEO', 'REGION_GEO', 'COMUNA_GEO']],
    on='NOMBRE_FULL_NORM',
    how='left'
)

print("=== MATCH EXACTO POR NOMBRE COMPLETO ===")
print("Total hospitales:", len(rh))
print("Sin coordenadas:", rh['LAT_GEO'].isna().sum())
print("Porcentaje:", round(rh['LAT_GEO'].isna().mean() * 100, 2), "%")

# =========================================================
# 7. MATCH EXACTO POR NOMBRE SIMPLE PARA FALTANTES
# =========================================================

faltantes_exacto = rh[rh['LAT_GEO'].isna()].copy()

aux_simple = faltantes_exacto.merge(
    de_geo_simple[['NOMBRE_SIMPLE_NORM', 'NOMBRE_GEO', 'LAT_GEO', 'LON_GEO', 'REGION_GEO', 'COMUNA_GEO']],
    on='NOMBRE_SIMPLE_NORM',
    how='left',
    suffixes=('', '_SIMPLE')
)

aux_simple.index = faltantes_exacto.index

rh.loc[faltantes_exacto.index, 'NOMBRE_GEO'] = (
    rh.loc[faltantes_exacto.index, 'NOMBRE_GEO']
    .combine_first(aux_simple['NOMBRE_GEO'])
)

rh.loc[faltantes_exacto.index, 'LAT_GEO'] = (
    rh.loc[faltantes_exacto.index, 'LAT_GEO']
    .combine_first(aux_simple['LAT_GEO'])
)

rh.loc[faltantes_exacto.index, 'LON_GEO'] = (
    rh.loc[faltantes_exacto.index, 'LON_GEO']
    .combine_first(aux_simple['LON_GEO'])
)

rh.loc[faltantes_exacto.index, 'REGION_GEO'] = (
    rh.loc[faltantes_exacto.index, 'REGION_GEO']
    .combine_first(aux_simple['REGION_GEO'])
)

rh.loc[faltantes_exacto.index, 'COMUNA_GEO'] = (
    rh.loc[faltantes_exacto.index, 'COMUNA_GEO']
    .combine_first(aux_simple['COMUNA_GEO'])
)

print("\n=== DESPUÉS DE EXACTO POR NOMBRE SIMPLE ===")
print("Sin coordenadas:", rh['LAT_GEO'].isna().sum())
print("Porcentaje:", round(rh['LAT_GEO'].isna().mean() * 100, 2), "%")


# =========================================================
# 7.5 EQUIVALENCIAS MANUALES PARA HOSPITALES GRANDES
# =========================================================

equivalencias_hospitales = {
    'HOSPITAL REGIONAL DE RANCAGUA': 'HOSPITAL REGIONAL DE RANCAGUA',
    'HOSPITAL DR FELIX BULNES CERDA (SANTIAGO, QUINTA NORMAL)': 'HOSPITAL DR FELIX BULNES CERDA',
    'HOSPITAL CLINICO REGIONAL (VALDIVIA)': 'HOSPITAL CLINICO REGIONAL VALDIVIA',
    'HOSPITAL DR JUAN NOE CREVANNI (ARICA)': 'HOSPITAL DR JUAN NOE CREVANI',
    'HOSPITAL CLINICO SAN BORJA ARRIARAN (SANTIAGO, SANTIAGO)': 'HOSPITAL CLINICO SAN BORJA ARRIARAN',
    'HOSPITAL DEL SALVADOR (SANTIAGO, PROVIDENCIA)': 'HOSPITAL DEL SALVADOR',
    'HOSPITAL SAN MARTIN (QUILLOTA)': 'HOSPITAL SAN MARTIN DE QUILLOTA',
    'HOSPITAL DE SAN CAMILO (SAN FELIPE)': 'HOSPITAL SAN CAMILO',
    'HOSPITAL DR ANTONIO TIRADO LANAS (OVALLE)': 'HOSPITAL ANTONIO TIRADO LANAS',
    'HOSPITAL SAN JUAN DE DIOS (SAN FERNANDO)': 'HOSPITAL SAN JUAN DE DIOS DE SAN FERNANDO',
    'HOSPITAL SAN JOSE (VICTORIA)': 'HOSPITAL SAN JOSE DE VICTORIA',
    'HOSPITAL DE NINOS DR LUIS CALVO MACKENNA (SANTIAGO, PROVIDENCIA)': 'HOSPITAL DR LUIS CALVO MACKENNA',
    'HOSPITAL REGIONAL (COIHAIQUE)': 'HOSPITAL REGIONAL COYHAIQUE'
}

rh['NOMBRE_MATCH_MANUAL'] = rh['NOMBRE_FULL_NORM'].replace(equivalencias_hospitales)

de_geo_manual = de_geo.rename(columns={
    'NOMBRE_FULL_NORM': 'NOMBRE_MATCH_MANUAL',
    'NOMBRE_GEO': 'NOMBRE_GEO_MANUAL',
    'LAT_GEO': 'LAT_GEO_MANUAL',
    'LON_GEO': 'LON_GEO_MANUAL',
    'REGION_GEO': 'REGION_GEO_MANUAL',
    'COMUNA_GEO': 'COMUNA_GEO_MANUAL'
})

rh = rh.merge(
    de_geo_manual[
        ['NOMBRE_MATCH_MANUAL', 'NOMBRE_GEO_MANUAL', 'LAT_GEO_MANUAL', 'LON_GEO_MANUAL', 'REGION_GEO_MANUAL', 'COMUNA_GEO_MANUAL']
    ].drop_duplicates(subset=['NOMBRE_MATCH_MANUAL']),
    on='NOMBRE_MATCH_MANUAL',
    how='left'
)

rh['LAT_GEO'] = rh['LAT_GEO'].fillna(rh['LAT_GEO_MANUAL'])
rh['LON_GEO'] = rh['LON_GEO'].fillna(rh['LON_GEO_MANUAL'])
rh['NOMBRE_GEO'] = rh['NOMBRE_GEO'].fillna(rh['NOMBRE_GEO_MANUAL'])
rh['REGION_GEO'] = rh['REGION_GEO'].fillna(rh['REGION_GEO_MANUAL'])
rh['COMUNA_GEO'] = rh['COMUNA_GEO'].fillna(rh['COMUNA_GEO_MANUAL'])

print("\n=== DESPUÉS DE EQUIVALENCIAS MANUALES ===")
print("Sin coordenadas:", rh['LAT_GEO'].isna().sum())
print("Porcentaje:", round(rh['LAT_GEO'].isna().mean() * 100, 2), "%")

# =========================================================
# 8. FUZZY POR NOMBRE COMPLETO
# =========================================================

def buscar_mejor_match_full(nombre):
    if pd.isna(nombre) or nombre is None or nombre == '':
        return (None, None)

    lista_geo = de_geo['NOMBRE_FULL_NORM'].dropna().unique().tolist()

    resultado = process.extractOne(
        nombre,
        lista_geo,
        scorer=fuzz.token_sort_ratio
    )

    if resultado is None:
        return (None, None)

    mejor_match, score, _ = resultado

#SCORE DE ACEPTACION
    if score >= 92:
        return (mejor_match, score)

    return (None, score)

faltantes = rh[rh['LAT_GEO'].isna()].copy()

resultado_fuzzy = faltantes['NOMBRE_FULL_NORM'].apply(buscar_mejor_match_full)

resultado_fuzzy = pd.DataFrame(
    resultado_fuzzy.tolist(),
    index=faltantes.index,
    columns=['NOMBRE_MATCH', 'SCORE_MATCH']
)

rh = rh.join(resultado_fuzzy)

de_geo_fuzzy = de_geo.rename(columns={
    'NOMBRE_FULL_NORM': 'NOMBRE_MATCH',
    'NOMBRE_GEO': 'NOMBRE_GEO_FUZZY',
    'LAT_GEO': 'LAT_GEO_FUZZY',
    'LON_GEO': 'LON_GEO_FUZZY',
    'REGION_GEO': 'REGION_GEO_FUZZY',
    'COMUNA_GEO': 'COMUNA_GEO_FUZZY'
})

rh = rh.merge(
    de_geo_fuzzy[
        ['NOMBRE_MATCH', 'NOMBRE_GEO_FUZZY', 'LAT_GEO_FUZZY', 'LON_GEO_FUZZY', 'REGION_GEO_FUZZY', 'COMUNA_GEO_FUZZY']
    ],
    on='NOMBRE_MATCH',
    how='left'
)

rh['LAT_GEO'] = rh['LAT_GEO'].fillna(rh['LAT_GEO_FUZZY'])
rh['LON_GEO'] = rh['LON_GEO'].fillna(rh['LON_GEO_FUZZY'])
rh['NOMBRE_GEO'] = rh['NOMBRE_GEO'].fillna(rh['NOMBRE_GEO_FUZZY'])
rh['REGION_GEO'] = rh['REGION_GEO'].fillna(rh['REGION_GEO_FUZZY'])
rh['COMUNA_GEO'] = rh['COMUNA_GEO'].fillna(rh['COMUNA_GEO_FUZZY'])

# =========================================================
# 9. RESULTADO FINAL
# =========================================================

print("\n=== DESPUÉS DE MATCH COMPLETO + SIMPLE + FUZZY ===")
print("Total hospitales:", len(rh))
print("Sin coordenadas:", rh['LAT_GEO'].isna().sum())
print("Porcentaje:", round(rh['LAT_GEO'].isna().mean() * 100, 2), "%")

no_match_final = rh[rh['LAT_GEO'].isna()].copy()
print("Pacientes en hospitales sin match:", no_match_final['total'].sum())
print(
    "Porcentaje de pacientes afectados:",
    round(no_match_final['total'].sum() / rh['total'].sum() * 100, 2),
    "%"
)

print("\n=== HOSPITALES QUE SIGUEN SIN MATCH ===")
display(
    no_match_final.sort_values('total', ascending=False)[
        ['NOMBRE_HOSPITAL', 'NOMBRE_FULL_NORM', 'NOMBRE_SIMPLE_NORM', 'NOMBRE_MATCH', 'total', 'porcentaje']
    ].head(30)
)

print("\n=== MATCHES FUZZY DUDOSOS ===")
dudosos = rh[rh['SCORE_MATCH'].notna() & (rh['SCORE_MATCH'] < 95)].copy()
display(
    dudosos.sort_values(['SCORE_MATCH', 'total'], ascending=[True, False])[
        ['NOMBRE_HOSPITAL', 'NOMBRE_FULL_NORM', 'NOMBRE_MATCH', 'NOMBRE_GEO_FUZZY', 'SCORE_MATCH', 'total']
    ].head(30)
)

resumen_hospital_geo = rh.copy()

=== MATCH EXACTO POR NOMBRE COMPLETO ===
Total hospitales: 71
Sin coordenadas: 15
Porcentaje: 21.13 %

=== DESPUÉS DE EXACTO POR NOMBRE SIMPLE ===
Sin coordenadas: 15
Porcentaje: 21.13 %

=== DESPUÉS DE EQUIVALENCIAS MANUALES ===
Sin coordenadas: 12
Porcentaje: 16.9 %

=== DESPUÉS DE MATCH COMPLETO + SIMPLE + FUZZY ===
Total hospitales: 71
Sin coordenadas: 10
Porcentaje: 14.08 %
Pacientes en hospitales sin match: 159479
Porcentaje de pacientes afectados: 15.89 %

=== HOSPITALES QUE SIGUEN SIN MATCH ===


,NOMBRE_HOSPITAL,NOMBRE_FULL_NORM,NOMBRE_SIMPLE_NORM,NOMBRE_MATCH,total,porcentaje
39,Hospital Regional de Rancagua,HOSPITAL REGIONAL DE RANCAGUA,DE RANCAGUA,NaN,26562,24.365635
21,"Hospital Dr. Félix Bulnes Cerda (Santiago, Qui...","HOSPITAL DR FELIX BULNES CERDA (SANTIAGO, QUIN...",FELIX BULNES CERDA,NaN,25387,24.465278
62,Hospital Clínico Regional (Valdivia),HOSPITAL CLINICO REGIONAL (VALDIVIA),NaN,NaN,23166,16.800483
0,Hospital Dr. Juan Noé Crevanni (Arica),HOSPITAL DR JUAN NOE CREVANNI (ARICA),JUAN NOE CREVANNI,NaN,19449,12.283408
27,"Hospital Del Salvador (Santiago, Providencia)","HOSPITAL DEL SALVADOR (SANTIAGO, PROVIDENCIA)",SALVADOR,NaN,18433,17.501221
13,Hospital San Martín (Quillota),HOSPITAL SAN MARTIN (QUILLOTA),SAN MARTIN,NaN,14011,19.606024
15,Hospital de San Camilo (San Felipe),HOSPITAL DE SAN CAMILO (SAN FELIPE),DE SAN CAMILO,NaN,13056,17.325368
8,Hospital Dr. Antonio Tirado Lanas (Ovalle),HOSPITAL DR ANTONIO TIRADO LANAS (OVALLE),ANTONIO TIRADO LANAS,NaN,11440,17.036713
29,Hospital de Niños Dr. Luis Calvo Mackenna (San...,HOSPITAL DE NINOS DR LUIS CALVO MACKENNA (SANT...,DE NINOS LUIS CALVO MACKENNA,NaN,7120,37.612360
65,Hospital Regional (Coihaique),HOSPITAL REGIONAL (COIHAIQUE),NaN,NaN,855,11.345029



=== MATCHES FUZZY DUDOSOS ===


,NOMBRE_HOSPITAL,NOMBRE_FULL_NORM,NOMBRE_MATCH,NOMBRE_GEO_FUZZY,SCORE_MATCH,total
62,Hospital Clínico Regional (Valdivia),HOSPITAL CLINICO REGIONAL (VALDIVIA),NaN,NaN,66.666667,23166
21,"Hospital Dr. Félix Bulnes Cerda (Santiago, Qui...","HOSPITAL DR FELIX BULNES CERDA (SANTIAGO, QUIN...",NaN,NaN,68.041237,25387
27,"Hospital Del Salvador (Santiago, Providencia)","HOSPITAL DEL SALVADOR (SANTIAGO, PROVIDENCIA)",NaN,NaN,69.767442,18433
13,Hospital San Martín (Quillota),HOSPITAL SAN MARTIN (QUILLOTA),NaN,NaN,71.186441,14011
29,Hospital de Niños Dr. Luis Calvo Mackenna (San...,HOSPITAL DE NINOS DR LUIS CALVO MACKENNA (SANT...,NaN,NaN,76.923077,7120
8,Hospital Dr. Antonio Tirado Lanas (Ovalle),HOSPITAL DR ANTONIO TIRADO LANAS (OVALLE),NaN,NaN,77.108434,11440
39,Hospital Regional de Rancagua,HOSPITAL REGIONAL DE RANCAGUA,NaN,NaN,77.551020,26562
15,Hospital de San Camilo (San Felipe),HOSPITAL DE SAN CAMILO (SAN FELIPE),NaN,NaN,85.294118,13056
0,Hospital Dr. Juan Noé Crevanni (Arica),HOSPITAL DR JUAN NOE CREVANNI (ARICA),NaN,NaN,87.804878,19449
65,Hospital Regional (Coihaique),HOSPITAL REGIONAL (COIHAIQUE),NaN,NaN,88.135593,855


## Sección 16: Auditoría de matches hospitalarios dudosos
La siguiente celda revisa hospitales sin coordenadas y propone candidatos por similitud, para control de calidad del matching y detección de casos que requieren validación manual.

In [28]:
###CODIO PARA REVISAR LOS DUDOSOS 

from rapidfuzz import process, fuzz

# ============================================
# AUDITORÍA DE CANDIDATOS FUZZY HOSPITALES
# ============================================

def mejor_candidato_hospital(nombre_objetivo, lista_candidatos):
    if pd.isna(nombre_objetivo) or nombre_objetivo is None or nombre_objetivo == '':
        return (None, None)

    resultado = process.extractOne(
        nombre_objetivo,
        lista_candidatos,
        scorer=fuzz.token_sort_ratio
    )

    if resultado is None:
        return (None, None)

    mejor_match, score, _ = resultado
    return (mejor_match, score)

lista_geo_full = de_geo['NOMBRE_FULL_NORM'].dropna().unique().tolist()

sin_match = resumen_hospital_geo[resumen_hospital_geo['LAT_GEO'].isna()].copy()

sin_match['MEJOR_CANDIDATO'] = sin_match['NOMBRE_FULL_NORM'].apply(
    lambda x: mejor_candidato_hospital(x, lista_geo_full)[0]
)

sin_match['MEJOR_SCORE'] = sin_match['NOMBRE_FULL_NORM'].apply(
    lambda x: mejor_candidato_hospital(x, lista_geo_full)[1]
)

revision_fuzzy = sin_match.merge(
    de_geo[['NOMBRE_FULL_NORM', 'NOMBRE_GEO', 'REGION_GEO', 'COMUNA_GEO']].drop_duplicates(subset=['NOMBRE_FULL_NORM']),
    left_on='MEJOR_CANDIDATO',
    right_on='NOMBRE_FULL_NORM',
    how='left',
    suffixes=('', '_CAND')
)

display(
    revision_fuzzy[
        [
            'NOMBRE_HOSPITAL',
            'NOMBRE_FULL_NORM',
            'MEJOR_CANDIDATO',
            'MEJOR_SCORE',
            'NOMBRE_GEO',
            'REGION_GEO',
            'COMUNA_GEO',
            'total'
        ]
    ].sort_values(['MEJOR_SCORE', 'total'], ascending=[False, False]).head(100)
)

,NOMBRE_HOSPITAL,NOMBRE_FULL_NORM,MEJOR_CANDIDATO,MEJOR_SCORE,NOMBRE_GEO,REGION_GEO,COMUNA_GEO,total
9,Hospital Regional (Coihaique),HOSPITAL REGIONAL (COIHAIQUE),HOSPITAL REGIONAL DE COYHAIQUE,88.135593,NaN,NaN,NaN,855
0,Hospital Dr. Juan Noé Crevanni (Arica),HOSPITAL DR JUAN NOE CREVANNI (ARICA),HOSPITAL REGIONAL DR JUAN NOE CREVANI (ARICA),87.804878,NaN,NaN,NaN,19449
3,Hospital de San Camilo (San Felipe),HOSPITAL DE SAN CAMILO (SAN FELIPE),HOSPITAL SAN CAMILO DE SAN FELIPE,85.294118,NaN,NaN,NaN,13056
7,Hospital Regional de Rancagua,HOSPITAL REGIONAL DE RANCAGUA,HOSPITAL DE NANCAGUA,77.551020,NaN,NaN,NaN,26562
1,Hospital Dr. Antonio Tirado Lanas (Ovalle),HOSPITAL DR ANTONIO TIRADO LANAS (OVALLE),HOSPITAL DR ANTONIO TIRADO LANAS DE OVALLE,77.108434,NaN,NaN,NaN,11440
6,Hospital de Niños Dr. Luis Calvo Mackenna (San...,HOSPITAL DE NINOS DR LUIS CALVO MACKENNA (SANT...,HOSPITAL DE NINOS DR LUIS CALVO MACKENNA,76.923077,NaN,NaN,NaN,7120
2,Hospital San Martín (Quillota),HOSPITAL SAN MARTIN (QUILLOTA),HOSPITAL SAN PABLO (COQUIMBO),71.186441,NaN,NaN,NaN,14011
5,"Hospital Del Salvador (Santiago, Providencia)","HOSPITAL DEL SALVADOR (SANTIAGO, PROVIDENCIA)","HOSPITAL EL PINO (SANTIAGO, SAN BERNARDO)",69.767442,NaN,NaN,NaN,18433
4,"Hospital Dr. Félix Bulnes Cerda (Santiago, Qui...","HOSPITAL DR FELIX BULNES CERDA (SANTIAGO, QUIN...","HOSPITAL EL PINO (SANTIAGO, SAN BERNARDO)",68.041237,NaN,NaN,NaN,25387
8,Hospital Clínico Regional (Valdivia),HOSPITAL CLINICO REGIONAL (VALDIVIA),HOSPITAL REGIONAL DR JUAN NOE CREVANI (ARICA),66.666667,NaN,NaN,NaN,23166


## Sección 17: Validación de cobertura georreferenciada
La siguiente celda cuantifica cuántos hospitales quedan efectivamente georreferenciados y revisa duplicidad por código, antes de pasar a la visualización definitiva.

In [29]:
# ============================================
# VALIDACIÓN DE HOSPITALES GEOREFERENCIADOS
# ============================================

hospitales_mapa_df = resumen_hospital_geo.copy()
hospitales_mapa_df = hospitales_mapa_df.dropna(subset=['LAT_GEO', 'LON_GEO']).copy()

print("Filas totales en hospitales_mapa_df:", len(hospitales_mapa_df))
print("COD_HOSPITAL únicos:", hospitales_mapa_df['COD_HOSPITAL'].nunique())

display(
    hospitales_mapa_df.groupby('COD_HOSPITAL')
    .size()
    .reset_index(name='filas_por_codigo')
    .sort_values('filas_por_codigo', ascending=False)
    .head(20)
)

# ============================================
# (DESPUÉS sigue tu código del mapa)
# ============================================

hospitales_mapa = hospitales_mapa_df.copy()
hospitales_mapa['porcentaje'] = hospitales_mapa['porcentaje'].round(2)

print("Hospitales con coordenadas:", len(hospitales_mapa))

Filas totales en hospitales_mapa_df: 61
COD_HOSPITAL únicos: 61


,COD_HOSPITAL,filas_por_codigo
0,102100,1
31,114105,1
33,115110,1
34,116100,1
35,116105,1
36,116107,1
37,116108,1
38,116110,1
39,116111,1
40,117101,1


Hospitales con coordenadas: 61


In [30]:
print(resumen_hospital_geo.columns.tolist())

['COD_HOSPITAL', 'NOMBRE_HOSPITAL', 'total', 'alta', 'porcentaje', 'NOMBRE_FULL_NORM', 'NOMBRE_SIMPLE_NORM', 'NOMBRE_GEO', 'LAT_GEO', 'LON_GEO', 'REGION_GEO', 'COMUNA_GEO', 'NOMBRE_MATCH_MANUAL', 'NOMBRE_GEO_MANUAL', 'LAT_GEO_MANUAL', 'LON_GEO_MANUAL', 'REGION_GEO_MANUAL', 'COMUNA_GEO_MANUAL', 'NOMBRE_MATCH', 'SCORE_MATCH', 'NOMBRE_GEO_FUZZY', 'LAT_GEO_FUZZY', 'LON_GEO_FUZZY', 'REGION_GEO_FUZZY', 'COMUNA_GEO_FUZZY']


In [31]:
# ============================================
# 1. Cantidad de hospitales únicos en cada base
# ============================================

print("Hospitales únicos en tabla maestra:", df_maestra['COD_HOSPITAL'].nunique())
print("Hospitales únicos en df_2024:", df_2024['COD_HOSPITAL'].nunique())
print("Hospitales únicos en resumen_hospital:", resumen_hospital['COD_HOSPITAL'].nunique())
print("Hospitales únicos en resumen_hospital_geo:", resumen_hospital_geo['COD_HOSPITAL'].nunique())

# ============================================
# 2. Ver qué hospitales del dataset NO están en la tabla maestra
# ============================================

codigos_df = set(pd.to_numeric(df_2024['COD_HOSPITAL'], errors='coerce').dropna().unique())
codigos_maestra = set(pd.to_numeric(df_maestra['COD_HOSPITAL'], errors='coerce').dropna().unique())

faltan_en_maestra = codigos_df - codigos_maestra

print("Códigos en df_2024 que no están en tabla maestra:", len(faltan_en_maestra))
print(sorted(list(faltan_en_maestra))[:50])

# ============================================
# 3. Ver qué hospitales del dataset quedaron sin coordenadas
# ============================================

hospitales_mapa = resumen_hospital_geo.copy()
hospitales_mapa_ok = hospitales_mapa.dropna(subset=['LAT_GEO', 'LON_GEO']).copy()

codigos_con_coord = set(hospitales_mapa_ok['COD_HOSPITAL'].dropna().unique())
codigos_sin_coord = codigos_df - codigos_con_coord

print("Hospitales únicos del dataset con coordenadas:", len(codigos_con_coord))
print("Hospitales únicos del dataset sin coordenadas:", len(codigos_sin_coord))
print(sorted(list(codigos_sin_coord))[:50])

# ============================================
# 4. Listado detallado de hospitales sin coordenadas
# ============================================

detalle_sin_coord = (
    resumen_hospital_geo[
        resumen_hospital_geo['COD_HOSPITAL'].isin(codigos_sin_coord)
    ][['COD_HOSPITAL', 'NOMBRE_HOSPITAL', 'REGION_GEO', 'COMUNA_GEO', 'total', 'porcentaje', 'LAT_GEO', 'LON_GEO']]
    .sort_values(['COD_HOSPITAL', 'REGION_GEO'])
)

display(detalle_sin_coord)

Hospitales únicos en tabla maestra: 377
Hospitales únicos en df_2024: 72
Hospitales únicos en resumen_hospital: 72
Hospitales únicos en resumen_hospital_geo: 71
Códigos en df_2024 que no están en tabla maestra: 1
[200717]
Hospitales únicos del dataset con coordenadas: 61
Hospitales únicos del dataset sin coordenadas: 11
[101100, 105102, 107101, 108100, 110120, 112100, 112102, 115100, 122100, 125100, 200717]


,COD_HOSPITAL,NOMBRE_HOSPITAL,REGION_GEO,COMUNA_GEO,total,porcentaje,LAT_GEO,LON_GEO
0,101100,Hospital Dr. Juan Noé Crevanni (Arica),NaN,NaN,19449,12.283408,NaN,NaN
8,105102,Hospital Dr. Antonio Tirado Lanas (Ovalle),NaN,NaN,11440,17.036713,NaN,NaN
13,107101,Hospital San Martín (Quillota),NaN,NaN,14011,19.606024,NaN,NaN
15,108100,Hospital de San Camilo (San Felipe),NaN,NaN,13056,17.325368,NaN,NaN
21,110120,"Hospital Dr. Félix Bulnes Cerda (Santiago, Qui...",NaN,NaN,25387,24.465278,NaN,NaN
27,112100,"Hospital Del Salvador (Santiago, Providencia)",NaN,NaN,18433,17.501221,NaN,NaN
29,112102,Hospital de Niños Dr. Luis Calvo Mackenna (San...,NaN,NaN,7120,37.612360,NaN,NaN
39,115100,Hospital Regional de Rancagua,NaN,NaN,26562,24.365635,NaN,NaN
62,122100,Hospital Clínico Regional (Valdivia),NaN,NaN,23166,16.800483,NaN,NaN
65,125100,Hospital Regional (Coihaique),NaN,NaN,855,11.345029,NaN,NaN


## Sección 18: Función de mapa comunal con hospitales (versión intermedia)
La siguiente celda redefine la función de visualización comunal para superponer puntos de hospitales sobre el coroplético, priorizando ubicación física reportada en la base geográfica.

In [32]:
# =========================================================
# BLOQUE NUEVO: DIBUJAR HOSPITALES GEOREFERENCIADOS EN MAPA COMUNAL
# =========================================================

# 1. Preparar DataFrame final para dibujo
hospitales_mapa_df = resumen_hospital_geo.copy()
hospitales_mapa_df = hospitales_mapa_df.dropna(subset=['LAT_GEO', 'LON_GEO']).copy()
hospitales_mapa_df['porcentaje'] = hospitales_mapa_df['porcentaje'].round(2)

# Normalizar solo columnas que existan
if 'REGION_GEO' in hospitales_mapa_df.columns:
    hospitales_mapa_df['REGION_GEO'] = (
        hospitales_mapa_df['REGION_GEO']
        .astype(str)
        .str.upper()
        .str.strip()
    )

if 'REGION' in hospitales_mapa_df.columns:
    hospitales_mapa_df['REGION'] = (
        hospitales_mapa_df['REGION']
        .astype(str)
        .str.upper()
        .str.strip()
    )

print("Filas georreferenciadas para dibujo:", len(hospitales_mapa_df))
print("Hospitales únicos georreferenciados:", hospitales_mapa_df['COD_HOSPITAL'].nunique())
print("Columnas disponibles:", hospitales_mapa_df.columns.tolist())


# 2. Redefinir función del mapa comunal incorporando hospitales
def mostrar_mapa_comunas(region_seleccionada, codregion):
    geojson_region = geojson_regiones.get(codregion)

    if geojson_region is None:
        print(f"No se encontró GeoJSON para la región {codregion}")
        return

    region_seleccionada = str(region_seleccionada).upper().strip()

    # Tabla comunal para coroplético
    tabla_comunas = resumen_comuna[resumen_comuna['REGION'] == region_seleccionada].copy()
    tabla_comunas['COMUNA_MAPA'] = (
        tabla_comunas['COMUNA']
        .astype(str)
        .str.title()
        .str.strip()
    )

    fig_comunas = px.choropleth(
        tabla_comunas,
        geojson=geojson_region,
        locations='COMUNA_GEOJSON',
        featureidkey='properties.Comuna',
        color='porcentaje',
        color_continuous_scale='OrRd',
        range_color=(0, 50),
        hover_name='COMUNA',
        custom_data=['total', 'alta'],
        title=f'Porcentaje de pacientes con alta severidad por comuna, {region_seleccionada} - 2024'
    )

    fig_comunas.update_geos(
        fitbounds="locations",
        visible=False
    )

    fig_comunas.update_layout(
        margin={"r": 20, "t": 60, "l": 20, "b": 20},
        paper_bgcolor="#F4F8FF",
        plot_bgcolor="#F4F8FF",
        title={
            "text": f"Porcentaje de pacientes con alta severidad por comuna, {region_seleccionada} - 2024",
            "x": 0.5,
            "xanchor": "center"
        }
    )

    fig_comunas.update_traces(
        hovertemplate=
        "<b>%{hovertext}</b><br>" +
        "Total pacientes: %{customdata[0]}<br>" +
        "Alta severidad: %{customdata[1]}<br>" +
        "Porcentaje: %{z:.2f}%<extra></extra>"
    )

    # Filtrar hospitales de la región seleccionada
    # Se privilegia REGION_GEO porque representa ubicación física
    if 'REGION_GEO' in hospitales_mapa_df.columns:
         hospitales_region = hospitales_mapa_df[
           hospitales_mapa_df['REGION_GEO'] == region_seleccionada
    ].copy()
    elif 'REGION' in hospitales_mapa_df.columns:
         hospitales_region = hospitales_mapa_df[
             hospitales_mapa_df['REGION'] == region_seleccionada
    ].copy()
    else:
        print("No existe columna REGION_GEO ni REGION en hospitales_mapa_df")
        hospitales_region = hospitales_mapa_df.iloc[0:0].copy()

    # Evitar duplicar puntos del mismo hospital
    # Nos quedamos con una fila por COD_HOSPITAL
    hospitales_region = (
        hospitales_region
        .sort_values('total', ascending=False)
        .drop_duplicates(subset=['COD_HOSPITAL'])
        .copy()
    )

    print(f"Hospitales a dibujar en {region_seleccionada}: {len(hospitales_region)}")

    # Dibujar hospitales sobre el mapa
    if len(hospitales_region) > 0:
        max_total = hospitales_region['total'].max()

        if max_total > 0:
            tamaños = (hospitales_region['total'] / max_total) * 18 + 6
        else:
            tamaños = [8] * len(hospitales_region)

        columnas_hover = ['total', 'alta', 'porcentaje']

        if 'COMUNA_GEO' in hospitales_region.columns:
            columnas_hover.append('COMUNA_GEO')
        else:
            hospitales_region['COMUNA_GEO'] = ''
            columnas_hover.append('COMUNA_GEO')

        fig_comunas.add_scattergeo(
            lon=hospitales_region['LON_GEO'],
            lat=hospitales_region['LAT_GEO'],
            text=hospitales_region['NOMBRE_HOSPITAL'],
            customdata=hospitales_region[columnas_hover],
            mode='markers',
            marker=dict(
                size=tamaños,
                color='blue',
                opacity=0.80,
                line=dict(width=0.7, color='white')
            ),
            name='Hospitales',
            hovertemplate=
            "<b>%{text}</b><br>" +
            "Comuna hospital: %{customdata[3]}<br>" +
            "Total pacientes: %{customdata[0]}<br>" +
            "Alta severidad: %{customdata[1]}<br>" +
            "Porcentaje: %{customdata[2]:.2f}%<extra></extra>"
        )

    fig_comunas.show()

Filas georreferenciadas para dibujo: 61
Hospitales únicos georreferenciados: 61
Columnas disponibles: ['COD_HOSPITAL', 'NOMBRE_HOSPITAL', 'total', 'alta', 'porcentaje', 'NOMBRE_FULL_NORM', 'NOMBRE_SIMPLE_NORM', 'NOMBRE_GEO', 'LAT_GEO', 'LON_GEO', 'REGION_GEO', 'COMUNA_GEO', 'NOMBRE_MATCH_MANUAL', 'NOMBRE_GEO_MANUAL', 'LAT_GEO_MANUAL', 'LON_GEO_MANUAL', 'REGION_GEO_MANUAL', 'COMUNA_GEO_MANUAL', 'NOMBRE_MATCH', 'SCORE_MATCH', 'NOMBRE_GEO_FUZZY', 'LAT_GEO_FUZZY', 'LON_GEO_FUZZY', 'REGION_GEO_FUZZY', 'COMUNA_GEO_FUZZY']


## Sección 19: Función final robusta para mapa comunal + hospitales
La siguiente celda implementa una versión más robusta de la visualización: normaliza nombres regionales, define equivalencias y presenta tanto el mapa como una tabla resumen de hospitales por región seleccionada.

In [33]:
# =========================================================
# BLOQUE NUEVO: DIBUJAR HOSPITALES EN MAPA COMUNAL (FINAL)
# =========================================================

import unicodedata
import re
import numpy as np

# 1. Copia final para dibujo

hospitales_mapa_df = resumen_hospital_geo.copy()
hospitales_mapa_df = hospitales_mapa_df.dropna(subset=['LAT_GEO', 'LON_GEO']).copy()
hospitales_mapa_df['porcentaje'] = hospitales_mapa_df['porcentaje'].round(2)

# 2. Función para normalizar texto
def normalizar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).upper().strip()
    texto = ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )
    texto = re.sub(r'[^A-Z0-9 ]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# 3. Normalizar columnas de región
if 'REGION_GEO' in hospitales_mapa_df.columns:
    hospitales_mapa_df['REGION_GEO_NORM'] = hospitales_mapa_df['REGION_GEO'].apply(normalizar_texto)
else:
    hospitales_mapa_df['REGION_GEO_NORM'] = ''

if 'REGION' in hospitales_mapa_df.columns:
    hospitales_mapa_df['REGION_NORM'] = hospitales_mapa_df['REGION'].apply(normalizar_texto)

# 4. Diccionario de equivalencias de región
mapa_region_geo = {
    'ARICA Y PARINACOTA': ['ARICA', 'PARINACOTA'],
    'TARAPACA': ['TARAPACA'],
    'ANTOFAGASTA': ['ANTOFAGASTA'],
    'ATACAMA': ['ATACAMA'],
    'COQUIMBO': ['COQUIMBO'],
    'VALPARAISO': ['VALPARAISO'],
    'OHIGGINS': ['OHIGGINS', 'LIBERTADOR'],
    'O HIGGINS': ['OHIGGINS', 'LIBERTADOR'],
    'MAULE': ['MAULE'],
    'NUBLE': ['NUBLE'],
    'BIOBIO': ['BIOBIO'],
    'ARAUCANIA': ['ARAUCANIA'],
    'LOS RIOS': ['LOS RIOS'],
    'LOS LAGOS': ['LOS LAGOS'],
    'AYSEN': ['AYSEN', 'AISEN'],
    'MAGALLANES': ['MAGALLANES'],
    'METROPOLITANA': ['METROPOLITANA', 'SANTIAGO']
}

# 5. FUNCIÓN FINAL DE MAPA
def mostrar_mapa_comunas_hospitales(region_seleccionada, codregion):

    geojson_region = geojson_regiones.get(codregion)

    if geojson_region is None:
        print(f"No se encontró GeoJSON para la región {codregion}")
        return

    # TABLA COMUNAL CORRECTA (USANDO GEO)
    tabla_comunas = resumen_comuna_match[
        resumen_comuna_match['codregion_geo'] == codregion
    ].copy()

    tabla_comunas = tabla_comunas.dropna(subset=['COMUNA_GEOJSON']).copy()

    tabla_comunas = (
        tabla_comunas.groupby(['COMUNA_GEOJSON'], as_index=False)
        .agg(
            total=('total', 'sum'),
            alta=('alta', 'sum')
        )
    )

    tabla_comunas['porcentaje'] = (
        tabla_comunas['alta'] / tabla_comunas['total'] * 100
    )

    tabla_comunas['COMUNA'] = tabla_comunas['COMUNA_GEOJSON']

    # MAPA COMUNAL
    fig_comunas = px.choropleth(
        tabla_comunas,
        geojson=geojson_region,
        locations='COMUNA_GEOJSON',
        featureidkey='properties.Comuna',
        color='porcentaje',
        color_continuous_scale='OrRd',
        range_color=(0, 50),
        hover_name='COMUNA',
        custom_data=['total', 'alta'],
        title=f'Porcentaje de pacientes con alta severidad por comuna, {region_seleccionada} - 2024'
    )

    fig_comunas.update_geos(
        fitbounds="locations",
        visible=False
    )

    fig_comunas.update_layout(
        margin={"r": 20, "t": 60, "l": 20, "b": 20},
        paper_bgcolor="#F4F8FF",
        plot_bgcolor="#F4F8FF",
        title={
            "text": f"Porcentaje de pacientes con alta severidad por comuna, {region_seleccionada} - 2024",
            "x": 0.5,
            "xanchor": "center"
        }
    )

    fig_comunas.update_traces(
        hovertemplate=
        "<b>%{hovertext}</b><br>" +
        "Total pacientes: %{customdata[0]}<br>" +
        "Alta severidad: %{customdata[1]}<br>" +
        "Porcentaje: %{z:.2f}%<extra></extra>"
    )

    # HOSPITALES
    region_norm = normalizar_texto(region_seleccionada)
    claves = mapa_region_geo.get(region_norm, [region_norm])

    mask_region_geo = hospitales_mapa_df['REGION_GEO_NORM'].apply(
        lambda x: any(clave in x for clave in claves)
    )

    hospitales_region = hospitales_mapa_df[mask_region_geo].copy()

    hospitales_region = (
        hospitales_region
        .sort_values('total', ascending=False)
        .drop_duplicates(subset=['COD_HOSPITAL'])
        .copy()
    )

    print(f"Región solicitada: {region_seleccionada}")
    print(f"Hospitales a dibujar: {len(hospitales_region)}")

    if len(hospitales_region) > 0:
        max_total = hospitales_region['total'].max()

        if max_total > 0:
            tamaños = (hospitales_region['total'] / max_total) * 18 + 6
        else:
            tamaños = np.repeat(8, len(hospitales_region))

        if 'COMUNA_GEO' not in hospitales_region.columns:
            hospitales_region['COMUNA_GEO'] = ''

        fig_comunas.add_scattergeo(
            lon=hospitales_region['LON_GEO'],
            lat=hospitales_region['LAT_GEO'],
            text=hospitales_region['NOMBRE_HOSPITAL'],
            customdata=hospitales_region[['COMUNA_GEO', 'total', 'alta', 'porcentaje', 'COD_HOSPITAL']],
            mode='markers',
            marker=dict(
                size=tamaños,
                color='blue',
                opacity=0.85,
                line=dict(width=0.8, color='white')
            ),
            name='Hospitales',
            hovertemplate=
            "<b>%{text}</b><br>" +
            "Código hospital: %{customdata[4]}<br>" +
            "Comuna hospital: %{customdata[0]}<br>" +
            "Total pacientes: %{customdata[1]}<br>" +
            "Alta severidad: %{customdata[2]}<br>" +
            "Porcentaje: %{customdata[3]:.2f}%<extra></extra>"
        )

    fig_comunas.show()

    columnas_tabla = [
        c for c in [
            'COD_HOSPITAL', 'NOMBRE_HOSPITAL', 'REGION_GEO', 'COMUNA_GEO',
            'LAT_GEO', 'LON_GEO', 'total', 'porcentaje'
        ]
        if c in hospitales_region.columns
    ]

    display(
        hospitales_region[columnas_tabla]
        .sort_values(['total', 'porcentaje'], ascending=[False, False])
        .head(30)
    )

## Sección 20: Dashboard interactivo nacional
La siguiente celda crea un tablero con clic en regiones. Al seleccionar una región, despliega automáticamente el mapa comunal y los hospitales correspondientes para análisis exploratorio guiado.

In [34]:
# BLOQUE NUEVO: MAPA DE CHILE INTERACTIVO
# CLICK EN REGIÓN -> MAPA COMUNAL + HOSPITALES


import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Output donde se mostrará el mapa comunal al hacer click
salida_region = widgets.Output()

# 2. Crear mapa regional de Chile como FigureWidget
fig_chile_click = go.FigureWidget(
    px.choropleth(
        resumen_region,
        geojson=geojson_chile,
        locations='codregion',
        featureidkey='properties.codregion',
        color='porcentaje',
        color_continuous_scale='Reds',
        hover_name='REGION',
        custom_data=['REGION', 'codregion', 'total', 'alta', 'porcentaje'],
        title='Porcentaje de pacientes con alta severidad por región, Chile 2024'
    )
)

fig_chile_click.update_geos(
    fitbounds="locations",
    visible=False
)

fig_chile_click.update_layout(
    margin={"r": 20, "t": 60, "l": 20, "b": 20},
    paper_bgcolor="#F4F8FF",
    plot_bgcolor="#F4F8FF",
    title={
        "text": "Porcentaje de pacientes con alta severidad por región, Chile 2024",
        "x": 0.5,
        "xanchor": "center"
    }
)

fig_chile_click.data[0].hovertemplate = (
    "<b>%{customdata[0]}</b><br>"
    "Total pacientes: %{customdata[2]}<br>"
    "Alta severidad: %{customdata[3]}<br>"
    "Porcentaje: %{customdata[4]:.2f}%<extra></extra>"
)

# 3. Función al hacer click sobre una región
def al_hacer_click_region(trace, points, state):
    if not points.point_inds:
        return
    
    idx = points.point_inds[0]
    
    fila = resumen_region.iloc[idx]
    region_sel = fila['REGION']
    codregion_sel = int(fila['codregion'])
    
    with salida_region:
        clear_output(wait=True)
        print(f"Región seleccionada: {region_sel} (código {codregion_sel})")
        mostrar_mapa_comunas_hospitales(region_sel, codregion_sel)

# 4. Vincular click
fig_chile_click.data[0].on_click(al_hacer_click_region)

# 5. Mostrar todo
display(fig_chile_click)
display(salida_region)

FigureWidget({
    'data': [{'coloraxis': 'coloraxis',
              'customdata': array([['ANTOFAGASTA', 2, 33581, 5435, 16.184747327357734],
                                   ['ARICA Y PARINACOTA', 15, 20144, 2580, 12.807783955520256],
                                   ['ATACAMA', 3, 20495, 2975, 14.515735545254941],
                                   ['AYSEN', 11, 906, 106, 11.699779249448124],
                                   ['BIOBIO', 8, 117156, 23450, 20.016046980094917],
                                   ['COQUIMBO', 4, 51412, 7966, 15.494437096397728],
                                   ['LA ARAUCANIA', 9, 79424, 13142, 16.546635777598713],
                                   ['LOS LAGOS', 10, 58867, 11294, 19.185621825470978],
                                   ['LOS RIOS', 14, 23547, 4013, 17.042510723234383],
                                   ['MAGALLANES', 12, 13427, 2668, 19.870410367170628],
                                   ['MAULE', 7, 81911, 14409, 17.5910439379

Output()

## Sección 20: Auditoría final de calidad para comunas
La siguiente celda reporta comunas sin match y comunas resueltas por fuzzy matching. Este control final permite transparentar calidad de integración geográfica antes de conclusiones.

In [35]:
# =========================================================
# AUDITORÍA REAL DEL MATCH DE COMUNAS
# =========================================================

print("Total filas en resumen_comuna_match:", len(resumen_comuna_match))
print("Con match:", resumen_comuna_match['COMUNA_GEOJSON'].notna().sum())
print("Sin match real:", resumen_comuna_match['COMUNA_GEOJSON'].isna().sum())

print("\n--- COMUNAS SIN MATCH REAL ---")
display(
    resumen_comuna_match[
        resumen_comuna_match['COMUNA_GEOJSON'].isna()
    ][['REGION', 'codregion', 'COMUNA', 'COMUNA_NORM', 'TIPO_MATCH', 'SCORE_MATCH']]
    .drop_duplicates()
    .sort_values(['REGION', 'COMUNA'])
)

print("\n--- COMUNAS RESUELTAS POR FUZZY ---")
display(
    resumen_comuna_match[
        resumen_comuna_match['TIPO_MATCH'] == 'fuzzy'
    ][['REGION', 'codregion', 'COMUNA', 'COMUNA_GEOJSON', 'SCORE_MATCH']]
    .drop_duplicates()
    .sort_values(['REGION', 'SCORE_MATCH'], ascending=[True, False])
)

Total filas en resumen_comuna_match: 597
Con match: 597
Sin match real: 0

--- COMUNAS SIN MATCH REAL ---


,REGION,codregion,COMUNA,COMUNA_NORM,TIPO_MATCH,SCORE_MATCH



--- COMUNAS RESUELTAS POR FUZZY ---


,REGION,codregion,COMUNA,COMUNA_GEOJSON,SCORE_MATCH
15,ANTOFAGASTA,2.0,MULCH�N,Mulchén,85.714286
4,ANTOFAGASTA,2.0,CA�ETE,Cañete,83.333333
16,ANTOFAGASTA,2.0,OLLAG�E,Ollagüe,71.428571
34,ARICA Y PARINACOTA,15.0,MACHAL�,Machalí,92.307692
68,AYSEN,11.0,COIHAIQUE,Coyhaique,88.888889
...,...,...,...,...,...
594,ÑUBLE,16.0,SAN NICOL�S,San Nicolás,90.909091
586,ÑUBLE,16.0,CONCEPCI�N,Concepción,90.000000
592,ÑUBLE,16.0,SAN FABI�N,San Fabián,90.000000
596,ÑUBLE,16.0,�IQU�N,Ñiquén,72.727273
